# Obtain Word Embeddings for Token Classifiers

### Table of Contents

[0.](#0) Preprocessing

[1.](#1) Custom with fastText

In [ ]:
import config
import pandas as pd
import numpy as np
import re, os
import my_utils
from gensim.models import FastText
from gensim.utils import tokenize
from gensim import utils
from gensim.test.utils import get_tmpfile

<a id="0"></a>
## 0. Preprocessing

In [ ]:
df_train = pd.read_csv(config.tokc_path+"model_input/token_train.csv", index_col=0)
df_dev = pd.read_csv(config.tokc_path+"model_input/token_validate.csv", index_col=0)
print(df_train.shape, df_dev.shape)
df_train.head()

Obtain the vocabulary of the annotated data:

In [ ]:
df = pd.concat([df_train, df_dev])  # df_train

In [ ]:
unique_tokens = list(set(list(df.token)))
unique_words = [token for token in unique_tokens if token.isalpha()]  # keep tokens with only alphabetic characters
print(len(unique_words), len(unique_tokens))

unique_tokens_lower = [token.lower() if token.isalpha() else token for token in unique_tokens]
unique_tokens_lower = list(set(unique_tokens_lower))
unique_words_lower = [token.lower() for token in unique_words]
unique_words_lower = list(set(unique_words_lower))
print(len(unique_words_lower), len(unique_tokens_lower))

<a id="1"></a>
## 1. Custom with fastText

Train custom word embeddings on my own data (metadata descriptions from the CRC's Archives catalog) using fastText.

* Data file: `data/descriptions_by_fonds`
* Date of harvesting: October 2020
* Harvesting and transformation code: [annot-prep/PreparationForAnnotation.ipynb](https://github.com/thegoose20/annot-prep/blob/main/PreparationForAnnotation.ipynb)

*References:* 
* *https://radimrehurek.com/gensim/models/fasttext.html*
* *https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html#sphx-glr-auto-examples-tutorials-run-fasttext-py*

In [ ]:
dir_path = config.docc_path+"model_input/"

In [ ]:
class CorpusIterator:
    def __iter__(self):
        file_list = ["train_docs.txt", "validate_docs.txt", "test_docs.txt"]
        for file_name in file_list:
            file_path = dir_path+file_name
            with utils.open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line != "|\n":
                        yield list(tokenize(line))  #yield list(tokenize(line.lower()))

Define the hyperparameters for the unsupervised training of the fastText model (essentially a word2vec model that uses using character n-grams so subwords can help to assign embeddings to unseen words):

In [ ]:
# Specify training architecture (default = "cbow" for Continuous Bag of Words)
models = ["cbow", "skipgram"]
model = models[0]
# Specify the learning rate (default = 0.025)
alpha = 0.025
# Specify the training objective (default = "ns")
# losses = ["ns", "hs", "softmax"]
# loss = losses[0]
# Specify the number of negative words to sample for 'ns' training objective (default = 5)
negative = 5
# Specify the threshold for downsampling higher-frequency words (default = 0.001)
sample = 0.001
# Specify the word embeddings' dimensions
vector_dimensions = 300 #50 #100
# Specify the context window (default is 5) 
context_window = 5
# Specify the number of epochs (default is 5)
epochs = 5
# Specify the threshold of word occurrences (ignore words that occur less than specified number of times; default = 5)
min_count = 5
# Specify the minimum and maximum length of character ngrams (defaults are 3 and 6)
min_n = 2
max_n = 6  # if 0, no character n-grams (subword vectors) will be used
# Specify the number of buckets for hashing ngrams (default = 2000000) 
bucket = 2000000
# Sort vocabulary by descending frequency (default = 1)
sorted_vocab = 1
# Specify the number of threads to use (default = 12)
# threads = 12

In [ ]:
model = FastText(
    alpha=alpha, negative=negative, sample=sample,
    vector_size=vector_dimensions, window=context_window, 
    epochs=epochs, min_count=min_count, min_n=min_n, 
    max_n=max_n, bucket=bucket, sorted_vocab=sorted_vocab
)

In [ ]:
model.build_vocab(corpus_iterable=CorpusIterator())
total_examples = model.corpus_count

In [ ]:
model.train(corpus_iterable=CorpusIterator(), total_examples=total_examples, epochs=epochs)
# Not lowercased, 300 dimensions: 
# Lowercased, 300 dimensions: (7321643, 10119275)
# Not lowercased, 200 dimensions: (7355581, 10119275)
# Lowercased, 200 dimensions: (7322249, 10119275)
# Not lowercased, 100 dimensions: (7321074, 10119275)
# Lowercased, 100 dimensions: (7322411, 10119275)
# Not lowercased, 50 dimensions: (7356468, 10119275)

In [ ]:
# model.wv["recipient"]

Save the model:

In [ ]:
# file_name = get_tmpfile(config.tokc_path+"fasttext100.model")
# file_name = get_tmpfile(config.tokc_path+"fasttext100_lowercased.model")
# file_name = config.tokc_path+"fasttext50.model"
# file_name = config.tokc_path+"fasttext50_lowercased.model"
# file_name = config.tokc_path+"fasttext50.model"
# file_name = config.tokc_path+"fasttext200.model"
# file_name = config.tokc_path+"fasttext200_lowercased.model"
file_name = config.tokc_path+"fasttext300.model"
# file_name = config.tokc_path+"fasttext300_lowercased.model"
model.save(file_name)

In [ ]:
len(model.wv) 
# Not lowercased, 300 dimensions:
# Lowercased, 300 dimensions: 17418
# Not lowercased, 200 dimensions: 20683
# Lowercased, 200 dimensions: 17418
# Not lowercased, 100 dimensions: 20683
# Lowercased, 100 dimensions: 17418
# Not lowercased, 50 dimensions: 20683
# Lowercased, 50 dimensions: 17418

In [ ]:
type(model.wv.key_to_index)     # Looks good

In [ ]:
"the" in model.wv.key_to_index  # Looks good

In [ ]:
"The" in model.wv.key_to_index  # Looks good